In [2]:
import os
import re
import glob
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
from sklearn.utils import resample
import urllib

In [ ]:
def parse_mpeg7_time(time_str):
    """Converts TRECVID time string (T00:00:17:1217F1499) to seconds."""
    m = re.match(r"T(\d+):(\d+):(\d+):(\d+)F(\d+)", time_str)
    if not m: return 0.0
    h, mn, s, frac, denom = map(int, m.groups())
    return h * 3600 + mn * 60 + s + frac / denom

def get_samples_from_collection(collection_xml_path, num_videos=3, skip_videos=0):
    """Parses <VideoFile> blocks to extract (video_id, download_url) pairs."""
    tree = ET.parse(collection_xml_path)
    root = tree.getroot()
    samples = []
    skipped = 0
    for vf in root.findall(".//VideoFile"):
        if skipped < skip_videos:
            skipped += 1
            continue
        video_id    = vf.find("id").text.strip()
        raw_fname   = vf.find("filename").text.strip()
        parts       = raw_fname.split("._-o-_.")
        filename    = parts[1] if len(parts) > 1 else parts[0]
        source_base = vf.find("source").text.strip()
        samples.append((video_id, f"{source_base}/{filename}"))
        if len(samples) >= num_videos:
            break
    return samples

def download_video(url, dest_path):
    """Downloads a video if not already present locally."""
    if os.path.exists(dest_path):
        print(f"  Already exists: {dest_path}")
        return True
    try:
        print(f"  Downloading: {url}")
        urllib.request.urlretrieve(url, dest_path)
        print("  Done.")
        return True
    except Exception as e:
        print(f"  Failed: {e}")
        return False

N_FEATS = 9

def compute_frame_pair_features(f1, f2):
    """
    Mixed features for a consecutive frame pair.
    f1, f2: BGR frames of the same dimensions.
    Returns float32 array of length N_FEATS.
    """
    h, w = f1.shape[:2]

    # Shared conversions reused by multiple features
    hsv1  = cv2.cvtColor(f1, cv2.COLOR_BGR2HSV)
    hsv2  = cv2.cvtColor(f2, cv2.COLOR_BGR2HSV)
    gray1 = cv2.cvtColor(f1, cv2.COLOR_BGR2GRAY).astype(np.float32)
    gray2 = cv2.cvtColor(f2, cv2.COLOR_BGR2GRAY).astype(np.float32)

    # --- 1. Bhattacharyya on 8x8x8 HSV histogram [28] ---
    hist1 = cv2.calcHist([hsv1], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    hist2 = cv2.calcHist([hsv2], [0, 1, 2], None, [8, 8, 8], [0, 180, 0, 256, 0, 256])
    cv2.normalize(hist1, hist1)
    cv2.normalize(hist2, hist2)
    bhatt = cv2.compareHist(hist1, hist2, cv2.HISTCMP_BHATTACHARYYA)

    # --- 2-4. Bogdanov [26]: 8x8 block brightness mean/std diffs ---
    GR, GC = 8, 8
    bh_g, bw_g = h // GR, w // GC
    m1, s1, m2, s2 = [], [], [], []
    for r in range(GR):
        for c in range(GC):
            b1 = gray1[r*bh_g:(r+1)*bh_g, c*bw_g:(c+1)*bw_g]
            b2 = gray2[r*bh_g:(r+1)*bh_g, c*bw_g:(c+1)*bw_g]
            m1.append(b1.mean()); s1.append(b1.std())
            m2.append(b2.mean()); s2.append(b2.std())
    mean_diffs = np.abs(np.array(m1) - np.array(m2))
    std_diffs  = np.abs(np.array(s1) - np.array(s2))
    bog_m1 = mean_diffs.mean()   # avg brightness diff (dark / light)
    bog_m2 = std_diffs.mean()    # avg texture diff    (calm / dynamic)
    bog_m3 = mean_diffs.max()    # peak brightness diff

    # --- 5-6. Sobel edge histogram over 10x10 = 100 blocks [9][10] ---
    edge1 = cv2.magnitude(cv2.Sobel(gray1, cv2.CV_32F, 1, 0),
                          cv2.Sobel(gray1, cv2.CV_32F, 0, 1))
    edge2 = cv2.magnitude(cv2.Sobel(gray2, cv2.CV_32F, 1, 0),
                          cv2.Sobel(gray2, cv2.CV_32F, 0, 1))
    BLK = 10
    bh_e, bw_e = h // BLK, w // BLK
    blk_diffs = np.array([
        abs(edge1[r*bh_e:(r+1)*bh_e, c*bw_e:(c+1)*bw_e].mean()
          - edge2[r*bh_e:(r+1)*bh_e, c*bw_e:(c+1)*bw_e].mean())
        for r in range(BLK) for c in range(BLK)
    ], dtype=np.float32)
    thresh     = 0.3 * blk_diffs.max() if blk_diffs.max() > 0 else 0.0
    edge_mean  = blk_diffs.mean()
    edge_count = float((blk_diffs > thresh).sum())

    # --- 7-8. Per-channel color histogram distance [27] ---
    N_BINS = 16
    ch_l1 = 0.0; ch_linf = 0.0
    for ch in range(3):
        h1c = cv2.calcHist([f1], [ch], None, [N_BINS], [0, 256]).flatten()
        h2c = cv2.calcHist([f2], [ch], None, [N_BINS], [0, 256]).flatten()
        h1c /= h1c.sum() + 1e-10
        h2c /= h2c.sum() + 1e-10
        d = np.abs(h1c - h2c)
        ch_l1  += d.sum()
        ch_linf = max(ch_linf, float(d.max()))
    color_l1   = ch_l1 / 3.0
    color_linf = ch_linf

    # --- 9. PyScene content score [29]: weighted delta-HSV ---
    dh = float(np.abs(hsv1[:,:,0].astype(np.float32) - hsv2[:,:,0].astype(np.float32)).mean()) / 180.0
    ds = float(np.abs(hsv1[:,:,1].astype(np.float32) - hsv2[:,:,1].astype(np.float32)).mean()) / 255.0
    dv = float(np.abs(hsv1[:,:,2].astype(np.float32) - hsv2[:,:,2].astype(np.float32)).mean()) / 255.0
    pyscene = 0.25 * dh + 0.25 * ds + 0.5 * dv

    return np.array([bhatt, bog_m1, bog_m2, bog_m3,
                     edge_mean, edge_count, color_l1, color_linf, pyscene],
                    dtype=np.float32)


def extract_features_and_labels(video_path, xml_path, window_size=5):
    """
    Reads a video and its MPEG-7 annotation.
    Returns X (N, 45) and y (N,) — mixed feature matrix and cut labels.
    """
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or not cap.isOpened():
        print(f"  Cannot open: {video_path}")
        cap.release()
        return None, None

    prev_frame = None
    raw_feats  = []

    while True:
        ret, frame = cap.read()
        if not ret: break
        small = cv2.resize(frame, (256, 144))
        if prev_frame is not None:
            raw_feats.append(compute_frame_pair_features(prev_frame, small))
        else:
            raw_feats.append(np.zeros(N_FEATS, dtype=np.float32))
        prev_frame = small

    cap.release()
    raw_feats  = np.array(raw_feats, dtype=np.float32)   # (N, 9)
    num_frames = len(raw_feats)

    y = np.zeros(num_frames, dtype=int)
    try:
        tree = ET.parse(xml_path)
        root = tree.getroot()
        ns   = {'ns': 'urn:mpeg:mpeg7:schema:2001'}
        segments = root.findall(".//ns:VideoSegment", ns)
        labeled  = 0
        for idx, seg in enumerate(segments):
            if idx == 0: continue
            te = seg.find(".//ns:MediaTimePoint", ns)
            if te is None:
                te = seg.find(".//ns:MediaRelTimePoint", ns)
            if te is not None and te.text:
                fi = int(round(parse_mpeg7_time(te.text.strip()) * fps))
                if fi < num_frames:
                    y[fi] = 1
                    labeled += 1
            else:
                print(f"  ⚠️ Segment {idx}: no time element — skipped.")
        if labeled == 0 and len(segments) > 1:
            print(f"  ⚠️ No boundaries labeled — check XML element names.")
        else:
            print(f"  Labeled {labeled} cut(s) from {len(segments)-1} segment(s).")
    except Exception as e:
        print(f"  ⚠️ XML parse error: {e}")
        return None, None

    pad    = np.zeros((window_size, N_FEATS), dtype=np.float32)
    padded = np.vstack([pad, raw_feats, pad]) 

    X = np.empty((num_frames, N_FEATS * 5), dtype=np.float32)
    for i in range(num_frames):
        w = padded[i : i + 2 * window_size + 1]
        X[i] = np.concatenate([raw_feats[i],
                                w.mean(axis=0), w.std(axis=0),
                                w.max(axis=0),  w.min(axis=0)])
    return X, y

In [18]:
COLLECTION_XML = "dataset2/iacc.3.collection.xml"

os.makedirs("dataset2/videos", exist_ok=True)

if not os.path.exists(COLLECTION_XML):
    raise FileNotFoundError(f"Missing: {COLLECTION_XML}")

print("Reading collection manifest...")
video_samples = get_samples_from_collection(COLLECTION_XML, num_videos=30, skip_videos=50)
for video_id, url in video_samples:
    download_video(url, f"dataset2/videos/{video_id}.mp4")

print(f"\nDone. Check dataset2/videos/ for downloaded files.")

Reading collection manifest...
  Downloading: http://archive.org/download/WestlessAmericanShort/WestlessAmericanShort_512kb.mp4
  Done.
  Downloading: http://archive.org/download/firemans_life/firemans_life_512kb.mp4
  Done.
  Downloading: http://archive.org/download/Betty_Boop_Training_Pigeons_1936/Betty_Boop_Training_Pigeons_1936_512kb.mp4
  Done.
  Downloading: http://archive.org/download/Musigreview-GeorgBreinschmidHommageToCharlesMingus702/Musigreview-GeorgBreinschmidHommageToCharlesMingus702_512kb.mp4
  Failed: HTTP Error 404: Not Found
  Downloading: http://archive.org/download/DiverDan_Ep06_Sawfish_Rescue/Diver_Dan_06_Sawfish_Rescue_DivX_DVD_640x448_b59__512kb.mp4
  Done.
  Downloading: http://archive.org/download/Dr_Peter_James-NightLandingInBudapest423-2/Dr_Peter_James-NightLandingInBudapest423_512kb.mp4
  Done.
  Downloading: http://archive.org/download/Dr_Peter_James-NightLandingInBudapest423/Dr_Peter_James-NightLandingInBudapest423_512kb.mp4
  Done.
  Downloading: http://a

KeyboardInterrupt: 

In [31]:
MP7_FOLDER  = "dataset2/mp7"
VIDEOS_DIR  = "dataset2/videos"
WINDOW_SIZE = 5
skip_processing = 0

skipped = 0
if not os.path.exists("train_data.pkl") or not os.path.exists("test_data.pkl"):
    X_global, y_global = [], []

    available = sorted(glob.glob(os.path.join(VIDEOS_DIR, "*.mp4")))
    print(f"Found {len(available)} video(s) in {VIDEOS_DIR}\n")

    for video_path in available:
        if skipped < skip_processing:
            skipped += 1
            print(f"Skipping {video_path} (skip count: {skipped})")
            continue
        video_id = os.path.splitext(os.path.basename(video_path))[0]
        xml_path = os.path.join(MP7_FOLDER, f"{video_id}.mp7.xml")

        if not os.path.exists(xml_path):
            print(f"Skipping {video_id}: no annotation at {xml_path}")
            continue

        print(f"Processing {video_id}...")
        X_vid, y_vid = extract_features_and_labels(video_path, xml_path, WINDOW_SIZE)
        if X_vid is not None:
            X_global.append(X_vid)
            y_global.append(y_vid)

    if not X_global:
        raise RuntimeError("No data collected. Ensure videos and mp7 annotations match by ID.")

    X_data = np.vstack(X_global)
    y_data = np.concatenate(y_global)

    print(f"\n--- Dataset ---")
    print(f"Frames: {X_data.shape[0]}  |  Features per frame: {X_data.shape[1]}")
    print(f"Shot cuts: {int(y_data.sum())} ({100.0 * y_data.mean():.2f}%)")

    
    ## merge to already existing train/test data if they exist
    
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y_data, test_size=0.20, stratify=y_data, random_state=42
    )
    if os.path.exists("train_data.pkl") and os.path.exists("test_data.pkl"):
        X_train_old, y_train_old = joblib.load("train_data.pkl")
        x_test_old, y_test_old = joblib.load("test_data.pkl")
        x_train = np.vstack([X_train_old, X_train])
        y_train = np.concatenate([y_train_old, y_train])
        x_test = np.vstack([x_test_old, X_test])
        y_test = np.concatenate([y_test_old, y_test])

    joblib.dump((X_train, y_train), "train_data.pkl")
    joblib.dump((X_test, y_test), "test_data.pkl")
else:
    X_train, y_train = joblib.load("train_data.pkl")
    X_test, y_test = joblib.load("test_data.pkl")

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

# save the features and labels for later use
print("\nSaved training data to train_data.pkl and test data to test_data.pkl")

#do the under sampling of the majority class to balance the dataset
from imblearn.under_sampling import RandomUnderSampler

print("\nUndersampling the majority class...")
rus = RandomUnderSampler(sampling_strategy=0.09,random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train)

print(f"Train (After Resampling): {X_train_resampled.shape[0]}")

rf = RandomForestClassifier(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
rf.fit(X_train_resampled, y_train_resampled)



Train: 416607  |  Test: 104152

Saved training data to train_data.pkl and test data to test_data.pkl

Undersampling the majority class...
Train (After Resampling): 65327


,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total n

In [32]:
y_pred = rf.predict(X_test)
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["No Cut", "Shot Cut"]))

joblib.dump(rf, "shot_detector_rf.pkl")
print("\nSaved model to shot_detector_rf.pkl")


--- Classification Report ---
              precision    recall  f1-score   support

      No Cut       0.99      0.99      0.99    102803
    Shot Cut       0.41      0.41      0.41      1349

    accuracy                           0.98    104152
   macro avg       0.70      0.70      0.70    104152
weighted avg       0.98      0.98      0.98    104152


Saved model to shot_detector_rf.pkl


In [ ]:
import numpy as np
import joblib

from lightgbm import LGBMClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

lgbm = LGBMClassifier(
    objective="binary",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

param_grid = {
    "num_leaves": [15, 31],
    "max_depth": [4, 6],
    "learning_rate": [0.03, 0.1],
    "n_estimators": [100, 300]
}

grid = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    verbose=2
)

print("Starting Grid Search...")
grid.fit(X_train_resampled, y_train_resampled)

print("\nBest Parameters:")
print(grid.best_params_)

print("\nBest CV F1:")
print(grid.best_score_)

best_model = grid.best_estimator_

print("\nEvaluating with threshold = 0.5")

y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred))

print("\nSearching for best threshold...")

proba = best_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.05, 0.95, 0.05)

best_threshold = 0.5
best_f1 = -1

for threshold in thresholds:

    pred = (proba >= threshold).astype(int)

    score = f1_score(y_test, pred)

    print(
        f"Threshold={threshold:.2f} "
        f"F1={score:.4f}"
    )

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("\nBest Threshold:", best_threshold)
print("Best F1:", best_f1)

final_pred = (proba >= best_threshold).astype(int)

print("\nFinal Classification Report")
print(classification_report(y_test, final_pred))

joblib.dump(best_model, "shot_detector_lgbm.pkl")


Starting Grid Search...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

Best Parameters:
{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'num_leaves': 31}

Best CV F1:
0.5737204760012181

Evaluating with threshold = 0.5
              precision    recall  f1-score   support

           0       0.99      0.99      0.99    102803
           1       0.36      0.45      0.40      1349

    accuracy                           0.98    104152
   macro avg       0.68      0.72      0.70    104152
weighted avg       0.98      0.98      0.98    104152


Searching for best threshold...


g:\Engineering\GP\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
g:\Engineering\GP\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Threshold=0.05 F1=0.0938
Threshold=0.10 F1=0.1625
Threshold=0.15 F1=0.2158
Threshold=0.20 F1=0.2557
Threshold=0.25 F1=0.2862
Threshold=0.30 F1=0.3169
Threshold=0.35 F1=0.3451
Threshold=0.40 F1=0.3690
Threshold=0.45 F1=0.3863
Threshold=0.50 F1=0.3993
Threshold=0.55 F1=0.4166
Threshold=0.60 F1=0.4218
Threshold=0.65 F1=0.4219
Threshold=0.70 F1=0.4225
Threshold=0.75 F1=0.4105
Threshold=0.80 F1=0.4040
Threshold=0.85 F1=0.3914
Threshold=0.90 F1=0.3655

Best Threshold: 0.7000000000000001
Best F1: 0.42254771415889925

Final Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    102803
           1       0.53      0.35      0.42      1349

    accuracy                           0.99    104152
   macro avg       0.76      0.67      0.71    104152
weighted avg       0.99      0.99      0.99    104152


Saved model to shot_detector_lgbm.pkl
